In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

df=pd.read_csv('telco_churn_cleaned.csv')
print("Shape:", df.shape)

Shape: (7043, 20)


In [2]:
df['Churn']=(df['Churn']=='Yes').astype(int)

#Encode binary columns
binary_mapping={
    'gender':{'Femal':0,'Male':1},
    'Partner':{'Yes':1,'No':0},
    'Dependents':{'Yes':1,'No':0},
    'PhoneService':{'Yes':1,'No':0},
    'PaperlessBilling':{'Yes':1,'No':0}
}                                                     

In [3]:
for col, mapping in binary_mapping.items():
    df[col]=df[col].map(mapping)

In [4]:
# One hot encode remaining columns
df = pd.get_dummies(df, columns=[
    'MultipleLines', 'InternetService',
    'OnlineSecurity', 'OnlineBackup',
    'DeviceProtection', 'TechSupport',
    'StreamingTV', 'StreamingMovies',
    'Contract', 'PaymentMethod'
], drop_first=True)

# Drop TotalCharges
df = df.drop('TotalCharges', axis=1)

# Split features and target
X = df.drop('Churn', axis=1)
y = df['Churn']

# Train test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Scale numerical columns
scaler = StandardScaler()
numerical_cols = ['tenure', 'MonthlyCharges']
X_train[numerical_cols] = scaler.fit_transform(
    X_train[numerical_cols])
X_test[numerical_cols] = scaler.transform(
    X_test[numerical_cols])

print("Data ready.")
print("Training set:", X_train.shape)
print("Test set:", X_test.shape)

Data ready.
Training set: (5634, 29)
Test set: (1409, 29)


In [5]:
import mlflow
import mlflow.sklearn

In [6]:
#Create or connect to an experiment
mlflow.set_experiment("churn_prediction")

print("MLflow experiment set up successfully.")
print("Tracking URI:", mlflow.get_tracking_uri())

MLflow experiment set up successfully.
Tracking URI: sqlite:///C:/Users/bharv/churn_pipeline/mlflow.db


In [7]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import(roc_auc_score,classification_report,f1_score, recall_score)

In [8]:
#Define model
rf_model= RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    class_weight='balanced',
    random_state=42
)

In [9]:
#Train

In [10]:
rf_model.fit(X_train,y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,10
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [11]:
#Predict
y_pred_rf= rf_model.predict(X_test)
y_pred_proba_rf=rf_model.predict_proba(X_test)[:,1]

In [12]:
#Calculate metrics
roc_auc_rf=roc_auc_score(y_test, y_pred_proba_rf)
f1_rf= f1_score(y_test, y_pred_rf)
recall_rf= recall_score(y_test, y_pred_rf)

In [13]:
#Log parameters to MLFlow
mlflow.log_param("n_estimators",100)
mlflow.log_param("max_depth",10)
mlflow.log_param("class_weight","balanced")

2026/05/23 19:38:41 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    export GIT_PYTHON_REFRESH=quiet



'balanced'

In [14]:
#Log metrics to MLflow
mlflow.log_metric("roc_auc", roc_auc_rf)
mlflow.log_metric("f1_score", f1_rf)
mlflow.log_metric("recall",recall_rf)

In [15]:
#Log model
mlflow.sklearn.log_model(rf_model,"random_forest_model")

print("random forest results:")
print(f"ROC-AUC: {round(roc_auc_rf, 4)}")
print(f"F1 Score: {round(recall_rf, 4)}")
print("\nClassification report:")
print(classification_report(y_test, y_pred_rf))

2026/05/23 19:38:42 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/23 19:38:42 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


random forest results:
ROC-AUC: 0.839
F1 Score: 0.7299

Classification report:
              precision    recall  f1-score   support

           0       0.89      0.77      0.83      1035
           1       0.54      0.73      0.62       374

    accuracy                           0.76      1409
   macro avg       0.71      0.75      0.72      1409
weighted avg       0.80      0.76      0.77      1409



In [16]:
mlflow.end_run()

In [23]:
import xgboost as xgb

mlflow.end_run()

with mlflow.start_run(run_name="XGBoost_baseline"):
    
    # Define model
    xgb_model = xgb.XGBClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=5,
        scale_pos_weight=len(y_train[y_train==0]) / 
                         len(y_train[y_train==1]),
        random_state=42,
        eval_metric='logloss',
        verbosity=0
    )
    
    # Train
    xgb_model.fit(X_train, y_train)
    
    # Predict
    y_pred_xgb = xgb_model.predict(X_test)
    y_pred_proba_xgb = xgb_model.predict_proba(X_test)[:, 1]
    
    # Calculate metrics
    roc_auc_xgb = roc_auc_score(y_test, y_pred_proba_xgb)
    f1_xgb = f1_score(y_test, y_pred_xgb)
    recall_xgb = recall_score(y_test, y_pred_xgb)
    
    # Log to MLflow
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("learning_rate", 0.1)
    mlflow.log_param("max_depth", 5)
    mlflow.log_metric("roc_auc", roc_auc_xgb)
    mlflow.log_metric("f1_score", f1_xgb)
    mlflow.log_metric("recall", recall_xgb)
    mlflow.sklearn.log_model(xgb_model, "xgboost_model")
    
    print("XGBoost Results:")
    print(f"ROC-AUC:  {round(roc_auc_xgb, 4)}")
    print(f"F1 Score: {round(f1_xgb, 4)}")
    print(f"Recall:   {round(recall_xgb, 4)}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred_xgb))

2026/05/23 19:42:38 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/23 19:42:38 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


XGBoost Results:
ROC-AUC:  0.8394
F1 Score: 0.627
Recall:   0.7888

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.74      0.81      1035
           1       0.52      0.79      0.63       374

    accuracy                           0.75      1409
   macro avg       0.71      0.76      0.72      1409
weighted avg       0.80      0.75      0.76      1409



In [24]:
print("Xgboost results")
print(f"ROC-AUC: {round(roc_auc_xgb,4)}")
print(f"F1 score: {round(f1_xgb, 4)}")
print(f"Recall: {round(recall_xgb,4)}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred_xgb))

Xgboost results
ROC-AUC: 0.8394
F1 score: 0.627
Recall: 0.7888

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.74      0.81      1035
           1       0.52      0.79      0.63       374

    accuracy                           0.75      1409
   macro avg       0.71      0.76      0.72      1409
weighted avg       0.80      0.75      0.76      1409



In [25]:
print(mlflow.get_experiment_by_name("churn_prediction"))

<Experiment: artifact_location='file:///C:/Users/bharv/churn_pipeline/mlruns/1', creation_time=1779574619475, experiment_id='1', last_update_time=1779574619475, lifecycle_stage='active', name='churn_prediction', tags={}, trace_location=None, workspace='default'>


In [26]:
from sklearn.model_selection import RandomizedSearchCV

mlflow.end_run()

with mlflow.start_run(run_name="XGBoost_tuned"):

    #Parameter grid to search over
    param_grid = {
        'n_estimators': [100,200,300,500],
        'learning_rate':[0.01,0.05,0.1,0.2],
        'max_depth':[3,4,5,6,7],
        'subsample':[0.7,0.8,0.9,1.0]
    }
    #Base model
    xgb_base=xgb.XGBClassifier(
    scale_pos_weight=len(y_train[y_train==0])/
                     len(y_train[y_train==1]),
    random_state=42,
    eval_metric='logloss',
    verbosity=0
    )

    #RandomizedSearchCV
    #n_iter=20-> try 20 random combinations
    #cv=5 -> 5 fold cross validation
    #scoring='roc_auc' -> optimize for ROC-AUC
    #n_jobs=-1 -> use all CPU cores to run faster
    random_search= RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_grid,
    n_iter=20,
    cv=5,
    scoring='roc_auc',
    n_jobs=-1,
    random_state=42,
    verbose=1
    )

    #Run the search
    print("Running hyperparameter tuning...")
    print("This will take 2-3 minutes...")
    random_search.fit(X_train, y_train)

    #Get best model
    best_xgb= random_search.best_estimator_

    #predict with best model
    y_pred_tuned = best_xgb.predict(X_test)
    y_pred_proba_tuned= best_xgb.predict_proba(X_test)[:,1]

    # Calculate metrics
    roc_auc_tuned = roc_auc_score(y_test, y_pred_proba_tuned)
    f1_tuned = f1_score(y_test, y_pred_tuned)
    recall_tuned = recall_score(y_test, y_pred_tuned)

    # Log best parameters to MLflow
    mlflow.log_params(random_search.best_params_)
    mlflow.log_metric("roc_auc", roc_auc_tuned)
    mlflow.log_metric("f1_score", f1_tuned)
    mlflow.log_metric("recall", recall_tuned)
    mlflow.log_metric("cv_best_score", random_search.best_score_)
    mlflow.sklearn.log_model(best_xgb, "xgboost_tuned_model")

    print("\nBest Parameters Found:")
    print(random_search.best_params_)
    print(f"\nCV Best ROC-AUC:  {round(random_search.best_score_, 4)}")
    print(f"Test ROC-AUC:     {round(roc_auc_tuned, 4)}")
    print(f"Test F1 Score:    {round(f1_tuned, 4)}")
    print(f"Test Recall:      {round(recall_tuned, 4)}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred_tuned))
    

Running hyperparameter tuning...
This will take 2-3 minutes...
Fitting 5 folds for each of 20 candidates, totalling 100 fits


2026/05/23 20:24:23 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/23 20:24:23 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html



Best Parameters Found:
{'subsample': 0.9, 'n_estimators': 100, 'max_depth': 3, 'learning_rate': 0.05}

CV Best ROC-AUC:  0.8485
Test ROC-AUC:     0.8458
Test F1 Score:    0.6276
Test Recall:      0.8155

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.72      0.80      1035
           1       0.51      0.82      0.63       374

    accuracy                           0.74      1409
   macro avg       0.71      0.77      0.72      1409
weighted avg       0.81      0.74      0.76      1409



In [27]:
import pickle

with open('best_xgb_model.pkl', 'wb') as f:
    pickle.dump(best_xgb, f)

with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print("Best model saved as best_xgb_model.pkl")
print("Scaler saved as scaler.pkl")

Best model saved as best_xgb_model.pkl
Scaler saved as scaler.pkl
